In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT_NAME = 'Arial'
SIG_FILL = 'C6EFCE'      # значимо (по Холму)
NOSIG_FILL = 'FCE4D6'    # не значимо (по Холму)
SKIP_FILL = 'D9D9D9'     # тест не посчитан (status != 'ok')
HEADER_FILL = '2F5496'


def export_results_to_excel(results_df, path='ab_test_bootstrap_report.xlsx',
                             metric_label='Результаты AB-теста (Poisson bootstrap)',
                             alpha=0.05):
    """
    results_df — результат bootstrap_ab_test.run_all_tests(df), обязательно
    с колонкой p_value_holm (holm_correction=True — это значение по умолчанию).
    Возвращает путь к сохранённому файлу.
    """
    if 'p_value_holm' not in results_df.columns:
        raise ValueError("В results_df нет колонки 'p_value_holm' — запусти "
                          "run_all_tests(..., holm_correction=True) (это значение по умолчанию)")

    has_channel = 'channel' in results_df.columns and (results_df['channel'] != 'ALL').any()

    # (заголовок, ширина колонки, числовой формат)
    columns = [('Кампания', 14, None)]
    if has_channel:
        columns.append(('Канал', 10, None))
    columns += [
        ('Сравнение', 16, None),
        ('n контр.', 9, None),
        ('n тест', 8, None),
        ('Ср. контр.', 11, '0.00'),
        ('Ср. тест', 10, '0.00'),
        ('Разница', 10, '0.00'),
        ('Uplift %', 9, '0.0%'),
        ('ДИ 2.5%', 10, '0.00'),
        ('ДИ 97.5%', 10, '0.00'),
        ('p-value', 10, '0.0000'),
        ('p-value (Холм)', 14, '0.0000'),
        ('Значимость', 26, None),
    ]
    headers = [c[0] for c in columns]
    widths = [c[1] for c in columns]
    formats = {i + 1: c[2] for i, c in enumerate(columns) if c[2]}
    ncol = len(headers)

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'AB-тест'

    thin = Side(style='thin', color='B7B7B7')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=ncol)
    ws.cell(1, 1, metric_label).font = Font(name=FONT_NAME, bold=True, size=14)

    header_row = 3
    for c, h in enumerate(headers, 1):
        cell = ws.cell(header_row, c, h)
        cell.font = Font(name=FONT_NAME, bold=True, color='FFFFFF')
        cell.fill = PatternFill('solid', fgColor=HEADER_FILL)
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
        cell.border = border

    sig_fill = PatternFill('solid', fgColor=SIG_FILL)
    nosig_fill = PatternFill('solid', fgColor=NOSIG_FILL)
    skip_fill = PatternFill('solid', fgColor=SKIP_FILL)

    data_start = header_row + 1
    for i, row in results_df.reset_index(drop=True).iterrows():
        r = data_start + i
        is_ok = row['status'] == 'ok'
        control_label = row['control'] if pd.notna(row['control']) else '—'
        comparison = f"{row['test']} vs {control_label}"

        vals = [row['brcb']]
        if has_channel:
            vals.append(row['channel'])
        vals.append(comparison)

        if is_ok:
            holm_sig = bool(row['p_value_holm'] < alpha)
            uplift_frac = row['uplift_pct'] / 100 if pd.notna(row['uplift_pct']) else None
            vals += [
                int(row['n_control']), int(row['n_test']),
                round(float(row['mean_control']), 2), round(float(row['mean_test']), 2),
                round(float(row['obs_diff']), 2),
                round(uplift_frac, 4) if uplift_frac is not None else None,
                round(float(row['ci_low']), 2), round(float(row['ci_high']), 2),
                round(float(row['p_value']), 4), round(float(row['p_value_holm']), 4),
                'Значимо' if holm_sig else 'Не значимо',
            ]
            fill = sig_fill if holm_sig else nosig_fill
        else:
            vals += [None] * 10 + [row['status']]
            fill = skip_fill

        for c, v in enumerate(vals, 1):
            cell = ws.cell(r, c, v)
            cell.font = Font(name=FONT_NAME)
            cell.border = border
            cell.fill = fill
            if v is not None and c in formats:
                cell.number_format = formats[c]

    for c, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(c)].width = w

    ws.freeze_panes = ws.cell(data_start, 1).coordinate

    wb.save(path)
    return path


# ------------------------------- пример запуска -------------------------------
# from bootstrap_ab_test import run_all_tests
# results_df = run_all_tests(df)
# export_results_to_excel(results_df, path='ab_test_bootstrap_report.xlsx')